# Xuan Thuy Prithvi-UNet: map, train and fine-tune

This notebook runs every feature of the `xuanthuy-prithvi-unet` package without writing new cells.

| I want to | Go to | GPU needed |
| --- | --- | --- |
| Set up a session (always first) | **1** | no |
| Check that the installation is correct | 2 | no |
| Check new images before using them | 3 | no |
| Make a land-cover map with a trained model | 4 | no (GPU is faster) |
| Score a map on the verified points | 4.4 | no |
| Train or fine-tune with other settings, bands or seeds | 5 | **yes** |
| Train on another area with its own labels | 5 (step 5.2) | **yes** |
| Score and compare checkpoints on the validation band | 6 | no (GPU is faster) |
| Download Sentinel-2 images for another area or period | 7 | no |
| Convert old checkpoints, print model shapes | 8 | no |
| Stop paying for the runtime | 9 | no |

**How to use it**

* Run section 1 at the start of every session (also after *Restart session* or a change of runtime type).
  Then run only the sections you need, top to bottom.
* Edit only the cells whose first line is `# EDIT`. Every other cell reads the variables set there.
* Results are written to Google Drive, so they survive the end of the session. Nothing is written into the repository.
* A training run lives in its own folder. Changing any setting means a new folder (the code refuses to mix settings).

**Where it runs**

| Where | How to start | Code runs on | Data |
| --- | --- | --- | --- |
| Colab in the browser | open the notebook, *Runtime > Change runtime type* | Colab VM | Google Drive |
| VS Code + Google Colab extension | *Select Kernel > Colab > New Colab Server* | Colab VM (not your PC) | Google Drive (`drive.mount` needs extension >= 0.2.1) |
| VS Code, local Python kernel | *Select Kernel > Python Environments* (venv with `pip install -e ".[export,notebook,test]"`) | your PC | local folders |

Documentation in the repository: `README.md` (overview), `docs/CODE_WALKTHROUGH.md` (what each file does),
`docs/DATA_AND_NORMALIZATION.md` (offset and normalisation), `docs/EXTENDING.md` (all extension points).

## 1. Session setup (run every session)

### 1.1 Runtime

Choose the runtime before running anything: *Runtime > Change runtime type*.

* Mapping, checks and scoring: CPU or the smallest GPU (T4) is enough.
* Training: a GPU. P3 (one date) uses about 7 GB of GPU memory with the default settings, so an L4 (24 GB) is enough;
  measure the speed in step 5.4 before choosing a larger GPU. Turn on *High-RAM* if available.

Changing the runtime type gives you a new machine: everything in `/content` is lost (cloned code, installed packages,
variables), files on Drive are kept. Run section 1 again afterwards.

In [ ]:
import os, sys, glob, json, subprocess, shutil, time, importlib
ON_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
print("Colab VM" if ON_COLAB else "local kernel", "| python", sys.version.split()[0])
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print("GPU:", gpu or "none (CPU runtime)")

### 1.2 Folders (edit once)

The project data comes with the repository (`data/`, see `data/README.md`): Sentinel-2 images of Xuan Thuy and
Thai Thuy, the label raster and the verified points. Google Drive keeps what must survive the session
(rename the folders here if yours differ):

```
MyDrive/Prithvi_Colab/
├── data/sentinel2_exports/    your own images (section 7), optional
├── weights/                   Prithvi_EO_V2_300M_TL.pt (downloaded once in 1.4)
├── configs_v1/                experiment configs written by section 5
├── runs_v1/                   one folder per training run (checkpoints, logs, map)
└── maps_v1/                   one folder per prediction (section 4)
```

In [ ]:
# EDIT -----------------------------------------------------------------------------------------
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                        # VS Code + Colab: command 'Colab: Mount Google Drive to Server...'
    BASE = "/content/drive/MyDrive/Prithvi_Colab"
    REPO = "/content/xuanthuy-prithvi-unet"
else:
    BASE = r"E:\Learning things\Aetosky\XuanthuyPrithvi"
    REPO = os.path.join(BASE, "xuanthuy-prithvi-unet")

DATA_ROOT = os.path.join(REPO, "data", "xuanthuy")            # in the repository: images, label raster, verified points
THAI_DATA = os.path.join(REPO, "data", "thaithuy")            # in the repository: images and verified points
EXPORTS = os.path.join(BASE, "data", "sentinel2_exports")      # your own images (section 7), optional
WEIGHTS = os.path.join(BASE, "weights", "Prithvi_EO_V2_300M_TL.pt")
CONFIGS_DIR = os.path.join(BASE, "configs_v1")
RUNS = os.path.join(BASE, "runs_v1")
MAPS = os.path.join(BASE, "maps_v1")
GIT_REF = "main"                                               # a tag (e.g. "v1.1.1") pins the exact code version
# ----------------------------------------------------------------------------------------------

for d in (CONFIGS_DIR, RUNS, MAPS):
    os.makedirs(d, exist_ok=True)
print(f"{'BASE':10} {'ok     ' if os.path.isdir(BASE) else 'MISSING'} {BASE}")
print(f"{'EXPORTS':10} {'ok     ' if os.path.isdir(EXPORTS) else 'not used yet (section 7)'} {EXPORTS}")
print(f"{'DATA_ROOT':10} {'ok     ' if os.path.isdir(DATA_ROOT) else 'comes with the clone in 1.3'} {DATA_ROOT}")
print(f"{'WEIGHTS':10} {'ok     ' if os.path.exists(WEIGHTS) else 'not yet (downloaded in 1.4)'} {WEIGHTS}")

### 1.3 Code and packages

The repository is private: the cell asks for a GitHub token with read access (classic PAT, scope `repo`, short expiry).
The token is typed, used once, and not stored in the notebook or in the clone.

If pip asks to restart the session (a library already loaded in memory was upgraded), restart and run 1.1–1.3 again;
the clone is kept, so the token is not asked twice.

In [ ]:
def git_rev(ref):
    out = subprocess.run(["git", "-C", REPO, "rev-parse", "--verify", "-q", ref + "^{commit}"], capture_output=True, text=True)
    return out.stdout.strip()

# A clone left on this machine by an earlier run is reused only if it is at GIT_REF; otherwise it is replaced.
if ON_COLAB and os.path.isdir(REPO) and (not git_rev(GIT_REF) or git_rev(GIT_REF) != git_rev("HEAD")):
    print(f"{REPO} is not at {GIT_REF}: cloning again")
    os.chdir("/content")
    shutil.rmtree(REPO)
    if "prithvi_unet" in sys.modules:
        print("An older prithvi_unet is already imported: after this cell, use Runtime > Restart session and run section 1 again.")
URL = "github.com/hongphuc-maker/xuanthuy-prithvi-unet.git"
if ON_COLAB and not os.path.isdir(REPO):
    import getpass
    token = getpass.getpass("GitHub token: ")
    subprocess.run(["git", "clone", "-q", f"https://x-access-token:{token}@{URL}", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://{URL}"], check=True)  # token not kept
    del token
    subprocess.run(["git", "-C", REPO, "checkout", "-q", GIT_REF], check=True)
os.chdir(REPO)
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[export,notebook,test]"], check=True)

# This kernel was started before `pip install -e`, so it does not see the package yet; the !python commands do.
src = os.path.join(REPO, "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

from importlib.metadata import version
import prithvi_unet
import numpy
if numpy.__version__ != version("numpy"):
    print(f"numpy {numpy.__version__} is loaded but pip installed {version('numpy')}: "
          "use Runtime > Restart session, then run 1.1-1.3 again (otherwise sections 5.3 and 6 fail inside numpy/scipy).")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
print("data:", {area: len(glob.glob(os.path.join(REPO, "data", area, "Sentinel2_*.tif"))) for area in ("xuanthuy", "thaithuy")},
      "images (expected 10 and 12)")
print("prithvi_unet", prithvi_unet.__version__, "|",
      {p: version(p) for p in ["torch", "terratorch", "timm", "numpy", "rasterio", "pandas"]})

### 1.4 Prithvi weights (downloaded to Drive once, 1.33 GB)

Needed only for training (section 5): a training run starts from these pretrained weights.
Mapping with a trained checkpoint does not need them.

In [ ]:
if not os.path.exists(WEIGHTS):
    from huggingface_hub import hf_hub_download
    hf_hub_download("ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL", "Prithvi_EO_V2_300M_TL.pt",
                    revision="63adbd39c271da4c42f447e69b1a7c91a338cdc9", local_dir=os.path.dirname(WEIGHTS))
os.environ["PRITHVI_WEIGHTS"] = WEIGHTS
print(WEIGHTS, round(os.path.getsize(WEIGHTS) / 1e9, 2), "GB")

### 1.5 Helper functions

Run once per session. Nothing is computed here; the functions are used by the later sections.

In [ ]:
import numpy as np, pandas as pd, rasterio, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, to_hex
from matplotlib.patches import Patch
from prithvi_unet.config import load_config, SENTINEL2_BANDS
from prithvi_unet.data.radiometry import prithvi_normalization
from prithvi_unet.utils import sha256_file

# Colours of the Xuan Thuy QGIS style (Xuanthuycolor.qml), class codes 1..11
XT_COLORS = ["#361fcb", "#d53216", "#0feb9a", "#6c90cf", "#99d17b", "#d6ed55",
             "#ce74c2", "#5cca6c", "#dbab52", "#9d41d2", "#d07997"]


def q(paths):
    """Quote one path or a list of paths for a ! shell command (Drive paths may contain spaces)."""
    return " ".join(f'"{p}"' for p in ([paths] if isinstance(paths, str) else paths))


def class_colors(class_map):
    k = len(class_map)
    return XT_COLORS if k == 11 else [to_hex(plt.get_cmap("tab20")(i % 20)) for i in range(k)]


def inspect_images(paths, bands=SENTINEL2_BANDS):
    """One row per image: the metadata the pipeline reads. Flags missing bands/dates and grid mismatches."""
    rows, grids = [], set()
    for p in paths:
        with rasterio.open(p) as src:
            tags, names = src.tags(), list(src.descriptions)
            grids.add((src.crs.to_string() if src.crs else None, src.width, src.height, tuple(src.transform)[:6]))
            rows.append({"file": os.path.basename(p), "acquisition_date": tags.get("acquisition_date", "MISSING"),
                         "bands_ok": set(bands) <= set(n for n in names if n),
                         "band_names": ",".join(n or "?" for n in names),
                         "boa_add_offset": tags.get("boa_add_offset", "-"),
                         "baseline": tags.get("s2_processing_baseline", "-"),
                         "size": f"{src.height} x {src.width}", "crs": str(src.crs), "res_m": src.res[0],
                         "dtype": src.dtypes[0], "valid_%": round(100 * float((src.dataset_mask() > 0).mean()), 1)})
    df = pd.DataFrame(rows)
    if df.empty:
        print("No image found: check the path and the file pattern.")
        return df
    if (df.acquisition_date == "MISSING").any() or not df.bands_ok.all():
        print("Some images lack band names or acquisition_date: add them with scripts/set_band_metadata.py (3.3).")
    print("All images on one grid." if len(grids) == 1 else
          f"{len(grids)} different grids: images used together (P2 dates) must share one grid.")
    return df


def show_rgb(path, ax=None, title=None, max_size=1200):
    """True-colour preview (B04, B03, B02), 2-98 % stretch, downsampled for speed."""
    with rasterio.open(path) as src:
        names = list(src.descriptions)
        scale = max(1, max(src.height, src.width) // max_size)
        shape = (src.height // scale, src.width // scale)
        rgb = np.stack([src.read(names.index(b) + 1, out_shape=shape).astype(np.float32) for b in ("B04", "B03", "B02")], -1)
    valid = rgb.sum(-1) > 0
    lo, hi = np.percentile(rgb[valid], 2, axis=0), np.percentile(rgb[valid], 98, axis=0)
    rgb = np.clip((rgb - lo) / np.maximum(hi - lo, 1), 0, 1)
    rgb[~valid] = 1
    ax = ax or plt.subplots(figsize=(8, 7))[1]
    ax.imshow(rgb); ax.set_title(title or os.path.basename(path), fontsize=9); ax.axis("off")
    return ax


def show_map(classes_tif, class_map, ax=None, title=None):
    """Class map with the legend and the share of each class (0 = no data, white)."""
    a = rasterio.open(classes_tif).read(1)
    colors = class_colors(class_map)
    ax = ax or plt.subplots(figsize=(9, 7))[1]
    ax.imshow(a, cmap=ListedColormap(["#ffffff"] + colors), vmin=0, vmax=len(colors), interpolation="nearest")
    v = a[a > 0]
    legend = [Patch(color=colors[k - 1], label=f"{k} {name}  {100 * (v == k).mean():.1f} %")
              for k, name in ((int(c), n) for c, n in class_map.items())]
    ax.legend(handles=legend, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=8)
    ax.set_title(title or os.path.basename(classes_tif), fontsize=9); ax.axis("off")
    return ax


def map_agreement(new_tif, old_tif, class_map, confidence_tif=None, top=5):
    """Agreement of two class maps of the same scene, class shares, most frequent swaps. Returns the agreement (0..1)."""
    a, b = rasterio.open(new_tif).read(1), rasterio.open(old_tif).read(1)
    names = {int(k): v for k, v in class_map.items()}
    if a.shape != b.shape:
        raise ValueError(f"Different sizes {a.shape} / {b.shape}: not the same scene")
    v = (a > 0) & (b > 0)
    print(f"no data new / old: {int((a == 0).sum()):,} / {int((b == 0).sum()):,}")
    print(f"agreement: {100 * (a[v] == b[v]).mean():.2f} % of {v.sum():,} pixels\n")
    print(pd.DataFrame({"new %": [100 * (a[v] == k).mean() for k in names],
                        "old %": [100 * (b[v] == k).mean() for k in names]}, index=list(names.values())).round(1))
    d = v & (a != b)
    swaps = pd.Series(list(zip(b[d], a[d]))).value_counts().head(top)
    print("\nmost frequent changes (old -> new, % of area):")
    for (o, n), count in swaps.items():
        print(f"  {names[int(o)]:8} -> {names[int(n)]:8} {100 * count / v.sum():.2f}")
    if confidence_tif:
        c = rasterio.open(confidence_tif).read(1)
        print(f"\nmedian max-softmax of the new map: where equal {np.median(c[v & (a == b)]):.2f}, "
              f"where different {np.median(c[d]):.2f}")
    return float((a[v] == b[v]).mean())


def set_by_path(cfg, dotted, value):
    """set_by_path(cfg, "training.seed", 1) -> cfg["training"]["seed"] = 1"""
    *parents, last = dotted.split(".")
    node = cfg
    for key in parents:
        node = node[key]
    if last not in node:
        print(f"note: '{dotted}' is a new key (check the spelling against the config)")
    node[last] = value


def write_config(cfg, name):
    """Save a config to CONFIGS_DIR and validate it with the package's own checks."""
    path = os.path.join(CONFIGS_DIR, f"{name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    load_config(path)  # raises a clear error if the config is inconsistent
    print("config written and validated:", path)
    return path


def config_from_checkpoint(checkpoint):
    """Configs saved inside checkpoints of this package (not inside old-repo checkpoints)."""
    import torch
    payload = torch.load(checkpoint, map_location="cpu", weights_only=False, mmap=True)
    cfg = payload.get("config") if isinstance(payload, dict) else None
    if cfg is None:
        return None
    path = os.path.join(CONFIGS_DIR, "_from_" + os.path.basename(os.path.dirname(checkpoint)) + ".json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    return path


def config_for_new_data(base_config, data_root, image_files, label_file, active_dates=None, class_map=None,
                        validation_columns=None, guard_px=48, rare_codes=None, experiment_id="new_area"):
    """Training config for other images and labels, starting from a repository config.

    image_files        all images of the area (GeoTIFFs inside data_root, one grid, band names + acquisition_date)
    label_file         one-band raster on the same grid: 0 = no label, 1..K = classes
    active_dates       dates fed to the model (default: the last T dates, T as in the base config)
    validation_columns (start, stop) pixel columns of the vertical validation band (default: ~62-76 % of the width)
    rare_codes         classes sampled with a quota (default: the three smallest classes)
    """
    cfg = load_config(base_config)
    t = len(cfg["active_dates"])
    images = []
    for f in image_files:
        with rasterio.open(f) as src:
            day = src.tags().get("acquisition_date")
            grid = (src.crs.to_string(), src.height, src.width, tuple(src.transform)[:6])
        if not day:
            raise ValueError(f"{f}: no acquisition_date tag (scripts/set_band_metadata.py)")
        images.append({"path": os.path.relpath(f, data_root).replace(os.sep, "/"), "date": day,
                       "sha256": sha256_file(f), "grid": grid})
    images.sort(key=lambda x: x["date"])
    if len({x["grid"] for x in images}) != 1:
        raise ValueError("All images must share one grid (CRS, size, transform)")
    crs, height, width, transform = images[0]["grid"]
    with rasterio.open(label_file) as src:
        if (src.crs.to_string(), src.height, src.width, tuple(src.transform)[:6]) != images[0]["grid"]:
            raise ValueError("The label raster is not on the image grid")
        label = src.read(1)
    class_map = class_map or cfg["class_map"]
    k = len(class_map)
    codes, counts = np.unique(label[label > 0], return_counts=True)
    if not set(codes.tolist()) <= set(range(1, k + 1)):
        raise ValueError(f"Label codes {codes.tolist()} are not within 1..{k} of the class map")
    if len(codes) < k:
        print(f"warning: classes {sorted(set(range(1, k + 1)) - set(codes.tolist()))} have no labelled pixel")
    start, stop = validation_columns or (int(0.62 * width), int(0.76 * width))
    rare = rare_codes if rare_codes is not None else sorted(int(c) for c in codes[np.argsort(counts)][:3])

    cfg["experiment_id"] = experiment_id
    cfg["images"] = [{k_: x[k_] for k_ in ("path", "date", "sha256")} for x in images]
    cfg["active_dates"] = active_dates or [x["date"] for x in images][-t:]
    cfg["target_date"] = cfg["active_dates"][-1]
    cfg["label"] = {"path": os.path.relpath(label_file, data_root).replace(os.sep, "/"),
                    "sha256": sha256_file(label_file), "reference_date": None, "status": "user supplied labels"}
    cfg["class_map"] = {str(k_): v for k_, v in class_map.items()}
    cfg["expected_shape"], cfg["expected_crs"] = [height, width], crs
    cfg["split"]["parameters"].update(validation_start=int(start), validation_stop=int(stop), guard_px_each_side=int(guard_px))
    cfg["split"]["sampling"].update(rare_codes=rare)
    cfg["acceptance"] = {"required_predicted_classes": k, "minimum_recall_by_code": {str(c): 0.05 for c in rare}}
    cfg.pop("independent_points", None)
    print(f"{len(images)} images, active dates {cfg['active_dates']}, grid {height} x {width} {crs}")
    print(f"validation columns {start}-{stop} (+{guard_px} px guard), rare classes {rare}")
    print("labelled pixels per class:", dict(zip(codes.tolist(), counts.tolist())))
    return cfg


def show_split(cfg, data_root):
    """Label raster with the validation band (blue) and its guard (grey)."""
    label = rasterio.open(os.path.join(data_root, cfg["label"]["path"])).read(1)
    p = cfg["split"]["parameters"]
    s, e, g = p["validation_start"], p["validation_stop"], p["guard_px_each_side"]
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(np.ma.masked_equal(label, 0), cmap=ListedColormap(class_colors(cfg["class_map"])),
              vmin=1, vmax=len(cfg["class_map"]), interpolation="nearest")
    ax.axvspan(s - g, s, color="grey", alpha=0.3); ax.axvspan(e, e + g, color="grey", alpha=0.3)
    ax.axvspan(s, e, color="tab:blue", alpha=0.25)
    ax.set_title("labels; blue = validation band, grey = guard (never used)", fontsize=9); ax.axis("off")
    print("labelled pixels inside the validation band:", int((label[:, s:e] > 0).sum()))

print("helpers ready")

## 2. Check the installation (optional, recommended once per code version)

* `pytest`: about 30 fast tests (radiometry, sampling, split, loss, metrics, shapes, band subsets) and the equivalence
  test against the model of the completed runs (same logits within 1e-4, same parameter counts).
  Expected: `N passed, 2 skipped` (the two skipped tests are the slow end-to-end run below).
  On v1.1.1 `test_no_project_file_is_gitignored` fails after `pip install -e` because it lists the generated
  `*.egg-info` folder; that test checks the repository's `.gitignore`, not the model.
* `trace_shapes.py`: every stage of the model with its output shape and parameter count (random weights, no download).
  Expected for P3: logits `(1, 11, 224, 224)`, 324,205,261 learned parameters.
* End-to-end (about 2 minutes per variant on CPU): train 1 step, validate, map and predict on synthetic images.

In [ ]:
!pytest -q
!python scripts/trace_shapes.py --config configs/p3_single_date.json

In [ ]:
!RUN_SLOW=1 pytest -q tests/test_end_to_end.py

## 3. Check images before using them

The model needs, for every image: the band names (`B02 B03 B04 B8A B11 B12`, or the subset of the config),
an `acquisition_date` tag, and the correct DN offset. Images from `scripts/export_sentinel2.py` (section 7) have all of
this. Images from other tools may not.

Why the offset matters: Sentinel-2 L2A products since 25 January 2022 store DN = 10000 x reflectance + 1000.
The reader subtracts the offset once, then standardises with the Prithvi pretraining mean and std.
A wrong offset shifts every band by 0.1 reflectance; the model still runs and silently gives a worse map.

In [ ]:
# EDIT -----------------------------------------------------------------------------------------
IMAGES = sorted(glob.glob(os.path.join(DATA_ROOT, "Sentinel2_2026-*.tif")))      # or THAI_DATA, or your EXPORTS folder
# ----------------------------------------------------------------------------------------------
inspect_images(IMAGES)

### 3.2 Offset and value range

`band_statistics.py` prints, per image, the offset it would use, then pooled percentiles of the offset-corrected values.

* **Pass:** in the per-band table the `p1` of B11 and B12 is a few tens at most (dark water reflects almost nothing in
  SWIR). For the Xuan Thuy 2026 images it is about 15-40.
* **Fail:** `p1` of B11/B12 near -1000: the images carry no offset (baseline before 04.00, or already harmonised data).
  Tag them with `--boa-add-offset 0` (3.3) or predict with `DN_OFFSET = 0`.
* A warning `offset chosen from the date` means the image has no offset tag (older exports); the offset then follows
  the acquisition date, which is correct for unprocessed L2A from Planetary Computer.
* `z_mean` is negative (about -0.2 to -0.8) for this coastal area: much water, darker than the global pretraining data.

In [ ]:
!python scripts/band_statistics.py {q(IMAGES)} --offset auto

### 3.3 Add missing metadata (only if 3.1 flagged problems)

Writes band names and tags **into the file** (in place). Work on a copy if the original must stay unchanged:
the checksum of the file changes, and configs that list it with a sha256 will refuse it.

In [ ]:
# EDIT and uncomment -----------------------------------------------------------------------------
# !python scripts/set_band_metadata.py "/path/to/image.tif" --bands B02 B03 B04 B8A B11 B12 --date 2026-10-15 --boa-add-offset -1000

### 3.4 Look at the images

In [ ]:
n = min(len(IMAGES), 6)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), squeeze=False)
for ax, path in zip(axes[0], IMAGES[-n:]):
    show_rgb(path, ax, title=os.path.basename(path)[10:20])
plt.tight_layout()

## 4. Map with a trained model (inference)

What happens: the scene is cut into 224 x 224 windows every 112 pixels, each window goes through the model,
the class probabilities of overlapping windows are averaged, and the most probable class is kept.
No labels are needed. Outputs, one GeoTIFF each, named after the date of the last image:

| File | Content |
| --- | --- |
| `<date>_classes.tif` | class code 1..K per pixel, 0 = no data |
| `<date>_max_softmax.tif` | probability of the chosen class (not calibrated) |
| `<date>_entropy.tif` | normalised entropy 0..1: high = the model hesitates between classes |
| `<date>_validity.tif` | 1 where the map is defined |

Rules:

* The config must match the checkpoint (same bands, same number of dates T). Checkpoints trained with this package
  carry their config: leave `PRED_CONFIG = None`. Checkpoints of the old repository (`P2_.../best_any.pt`,
  `P3_.../best_any.pt`) need it set: `configs/p3_single_date.json` or `configs/p2_multitemporal.json`.
* P3 (single date) takes 1 image. P2 (multi-temporal) takes 6 images; they are sorted by date and the last one is the
  date of the map. Use a seasonal spread similar to training (January, May, July, August, September, target).
* Accuracy on another area or year is unknown without labels: look at the map and the entropy before using it.

In [ ]:
# EDIT -----------------------------------------------------------------------------------------
CHECKPOINT = os.path.join(RUNS, "p3_repro", "best_any.pt")          # or BASE + "/runs/P3_single_sep21_UNet_WCE_REPRO/best_any.pt"
PRED_CONFIG = None                                                   # None = config stored in the checkpoint
PRED_IMAGES = sorted(glob.glob(os.path.join(DATA_ROOT, "Sentinel2_2026-10-01_*.tif")))
OUT_NAME = "p3_repro_xuanthuy_20261001"                             # folder inside MAPS
DN_OFFSET = None        # None = as in the config (-1000); "auto" = per image from its tags/date; 0 = no offset
UPSAMPLE = None         # None = bilinear, as trained; "nearest" only to study the effect of the final resize
# ----------------------------------------------------------------------------------------------

pred_config = PRED_CONFIG or config_from_checkpoint(CHECKPOINT)
if pred_config is None:
    raise ValueError("This checkpoint has no stored config (old repository): set PRED_CONFIG")
frames = len(load_config(pred_config)["active_dates"])
if len(PRED_IMAGES) != frames:
    raise ValueError(f"This model takes {frames} image(s); PRED_IMAGES has {len(PRED_IMAGES)}")
OUT_DIR = os.path.join(MAPS, OUT_NAME)
options = (f" --dn-offset {DN_OFFSET}" if DN_OFFSET is not None else "") + (f" --upsample {UPSAMPLE}" if UPSAMPLE else "")
images_arg = q(PRED_IMAGES)
t0 = time.time()
!python scripts/predict.py --config "{pred_config}" --checkpoint "{CHECKPOINT}" --images {images_arg} --out "{OUT_DIR}" {options}
print(f"{time.time() - t0:.0f} s")

### 4.2 Look at the map

Left: the last input image. Middle: the map with the share of each class. Right: entropy, bright where the model
hesitates (class boundaries, clouds, classes it rarely saw). The tags record the checkpoint, its sha256, dates, offset
and bands, so a map can always be traced back to what produced it.

In [ ]:
pred_cfg = load_config(pred_config)
classes_tif = sorted(glob.glob(os.path.join(OUT_DIR, "*_classes.tif")))[-1]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
show_rgb(PRED_IMAGES[-1], axes[0], "input (true colour)")
show_map(classes_tif, pred_cfg["class_map"], axes[1], os.path.basename(classes_tif))
entropy = rasterio.open(classes_tif.replace("_classes", "_entropy")).read(1)
im = axes[2].imshow(entropy, cmap="magma", vmin=0, vmax=1); axes[2].set_title("normalised entropy", fontsize=9)
axes[2].axis("off"); plt.colorbar(im, ax=axes[2], fraction=0.035)
plt.tight_layout()
print({k: v for k, v in rasterio.open(classes_tif).tags().items() if k != "AREA_OR_POINT"})

### 4.3 Compare with another map of the same scene (optional)

Use it to compare two models (P2 and P3, two training runs, two seeds) or to check that a re-run reproduces an older
map. Reading the output:

* Same checkpoint, same images: agreement should be above 99.9 %. Lower means different inputs or radiometry.
* Two independent training runs: expect about 90 % or more. If the differences are swaps in both directions between
  similar classes and the max-softmax is low where they differ, they are run-to-run noise. A one-way change of a whole
  class (for example all open water from one class to another) is a systematic difference worth investigating.

In [ ]:
# EDIT -----------------------------------------------------------------------------------------
OTHER_MAP = os.path.join(BASE, "single_inference_runs/protocol_A/xuan_thuy_2026_oct01_single/d4828f29f4ed/classes_20261001.tif")
# ----------------------------------------------------------------------------------------------
map_agreement(classes_tif, OTHER_MAP, pred_cfg["class_map"], classes_tif.replace("_classes", "_max_softmax"))

### 4.4 Agreement with the verified points

The repository holds verified points of Xuan Thuy (1,037 points, 7 classes) and Thai Thuy (616 points, 6 classes),
one shapefile per class. Each point is looked up in its pixel; OA is the share of points where the map gives the
point's class, macro-F1 is averaged over the classes that have points.

* Score maps of a date **not used in training** (1 Oct for Xuan Thuy). 886 of the 1,037 Xuan Thuy points lie in the
  training area, so on the training image (21 Sep) every model looks better than it is.
* The date of the points is not confirmed: report the result as agreement, not as the accuracy of a dated map.
* Dc, DK, trống and Gt have no points. Never use these scores to choose a checkpoint (section 6 does that).

Reference values on 1 Oct 2026: U-Net 0.661, P2 0.875, P3 completed run 0.859, P3 re-run 0.866.

In [ ]:
# EDIT -----------------------------------------------------------------------------------------
POINTS = os.path.join(DATA_ROOT, "verified_points_shapefiles.zip")   # Thai Thuy: os.path.join(THAI_DATA, ...)
MAPS_TO_SCORE = [classes_tif]                                         # add other *_classes.tif of the same area
# ----------------------------------------------------------------------------------------------
!python scripts/score_points.py --config "{pred_config}" --points "{POINTS}" --maps {q(MAPS_TO_SCORE)} --out "{OUT_DIR}/points"

## 5. Train or fine-tune

A run fine-tunes the whole model: the Prithvi encoder starts from its pretrained weights (learning rate 1e-5),
the feature pyramid, U-Net decoder and head start from random weights (learning rate 1e-4). Every 156 steps the model
is scored on the validation band; the checkpoint with the highest macro-F1 is kept as `best_any.pt`; training stops
after 6 validations without a gain of 0.001, or at 3,120 steps. At the end the map of the target date is written.

Steps: **5.1** (or 5.2 for new data) write a config → **5.3** prepare and audit the data → **5.4** short test and time
estimate → **5.5** full run → **5.6** results.

### 5.1 Config: start from a repository config and change settings

Base configs: `p3_single_date.json` (1 date), `p2_multitemporal.json` (6 dates),
`p3_single_date_4bands.json` (B02 B03 B04 B8A), `p3_single_date_3bands.json` (B02 B03 B04).

| Setting | Default | Effect |
| --- | --- | --- |
| `training.seed` | 20260910 | weight initialisation, dropout, batch order. Repeat a run with 2-3 seeds before comparing models: two runs of the same config differed by 0.02 macro-F1 |
| `training.freeze_encoder` | false | true = train only pyramid, decoder and head (20.3 M of 324 M parameters): measures how good the pretrained features are |
| `training.encoder_lr`, `training.decoder_lr` | 1e-5, 1e-4 | learning rates of the pretrained and the new layers |
| `training.maximum_steps`, `training.patience_evaluations` | 3120, 6 | length of the run and early stopping |
| `training.micro_batch_size` | 1 | images per forward pass; the batch stays 16 and the loss is the same; larger = faster, more GPU memory |
| `model.gradient_checkpointing` | true | false = faster, more GPU memory, same numbers |
| `model.decoder_channels`, `model.head_dropout` | [512, 256, 128, 64], 0.1 | size of the decoder, dropout before the classifier |
| `model.out_indices` | [5, 11, 17, 23] | encoder blocks feeding the 4 pyramid levels |
| `bands` | 6 bands | any subset of the six; the normalisation is adjusted automatically below |
| `split.parameters` | columns 1122-1388, guard 48 | validation band; changing it changes what the scores mean |

More: `docs/EXTENDING.md`, section 2.

In [ ]:
# EDIT -----------------------------------------------------------------------------------------
BASE_CONFIG = "configs/p3_single_date.json"
EXPERIMENT_ID = "P3_seed20260911"                      # names the config and the run folder; one id per experiment
CHANGES = {
    "training.seed": 20260911,
    # "training.freeze_encoder": True,
    # "training.micro_batch_size": 4,
    # "model.gradient_checkpointing": False,
    # "bands": ["B02", "B03", "B04", "B8A"],
}
# ----------------------------------------------------------------------------------------------

cfg = load_config(BASE_CONFIG)
for key, value in CHANGES.items():
    set_by_path(cfg, key, value)
if "bands" in CHANGES:                                   # one mean/std per band, from the Prithvi statistics
    cfg["normalization"].update(prithvi_normalization(cfg["bands"]))
cfg["experiment_id"] = EXPERIMENT_ID
TRAIN_CONFIG = write_config(cfg, EXPERIMENT_ID)
RUN_DIR = os.path.join(RUNS, EXPERIMENT_ID)
print("run folder:", RUN_DIR, "(exists: training will resume)" if os.path.isdir(RUN_DIR) else "(new)")

### 5.2 Config for other images or another area (instead of 5.1)

Requirements: all images inside `TRAIN_DATA_ROOT`, on one grid, with band names and `acquisition_date` (section 3);
a label raster on the same grid (one band, 0 = no label, 1..K = classes).

Choices that change the meaning of the results:

* **Validation band.** Pixels next to each other look alike, so a random pixel split gives optimistic scores.
  The validation band is a strip of columns; a guard on each side is never used; training windows never touch the band.
  Pick a strip that contains every class.
* **Rare classes** get a fixed number of training windows each (`rare_quota_each`, 416) and appear in every batch.
* **`n_patches` (4,992)** must be smaller than the number of possible training windows; for a small area reduce it
  (the error in 5.3 states how many windows are possible).
* With another class map, give `CLASS_MAP` as `{1: "name", 2: "name", ...}`; the model gets one output per class.

In [ ]:
# EDIT and run only for new data ---------------------------------------------------------------
TRAIN_DATA_ROOT = os.path.join(BASE, "data", "my_area")
EXPERIMENT_ID = "my_area_P3"
cfg = config_for_new_data(
    base_config="configs/p3_single_date.json",          # sets T, bands, model and training settings
    data_root=TRAIN_DATA_ROOT,
    image_files=sorted(glob.glob(os.path.join(TRAIN_DATA_ROOT, "Sentinel2_*.tif"))),
    label_file=os.path.join(TRAIN_DATA_ROOT, "labels.tif"),
    active_dates=None,                                  # None = the last date (P3) / last six dates (P2)
    class_map=None,                                     # None = the 11 Xuan Thuy classes
    validation_columns=None,                            # e.g. (900, 1100); None = about 62-76 % of the width
    rare_codes=None,                                    # None = the three smallest classes
    experiment_id=EXPERIMENT_ID,
)
# cfg["split"]["sampling"]["n_patches"] = 2000           # small area
show_split(cfg, TRAIN_DATA_ROOT)
TRAIN_CONFIG = write_config(cfg, EXPERIMENT_ID)
DATA_ROOT = TRAIN_DATA_ROOT                            # used by 5.3-5.5
RUN_DIR = os.path.join(RUNS, EXPERIMENT_ID)

### 5.3 Prepare and audit the data (CPU, a few minutes)

Checks every checksum, builds the validity mask (valid on every listed image), the split, the training windows
and the batch schedule, and writes them to `RUN_DIR/prepared/` with their checksums. The full run reuses them.
It also writes `RUN_LOCK.json`: the config and library versions of this run. If they change later (another config,
or Colab updated torch), the run folder refuses to continue; use a new `EXPERIMENT_ID`.

Expected for the Xuan Thuy configs: 294,156 validation pixels, 4,992 windows, 312 batches, 36 validation windows,
offset -1000. The warning `No per-pixel cloud mask` is expected: clouds are not masked.

In [ ]:
from prithvi_unet.training import Trainer
trainer = Trainer(TRAIN_CONFIG, DATA_ROOT, RUN_DIR)
audit = json.load(open(os.path.join(RUN_DIR, "prepared", "DATA_AUDIT.json")))
print({k: audit[k] for k in ["train_pixels", "validation_pixels", "n_patches", "n_batches", "n_validation_windows"]})
print("images (date, offset, valid pixels):", [(i["date"], i["dn_offset"], i["valid_pixels"]) for i in audit["images"]])
names = list(trainer.config["class_map"].values())
print(pd.DataFrame({"train pixels": audit["train_class_counts"], "loss weight": np.round(audit["class_weights"], 3)}, index=names))
del trainer                                          # frees memory; the prepared files stay on Drive

### 5.4 Short test: 20 steps, time and memory (GPU)

Runs 20 optimiser steps in a scratch folder (deleted and recreated each time), loads the pretrained weights,
and estimates the length of the full run.

* **Pass:** no error; `train_WCE` finite, starting near 2 and decreasing; `grad_norm` finite.
* Seconds per step on an L4 for P3 with default settings: about 2.2. The estimate does not include validation;
  the measured full P3 run took about 1.5 hours for 2,028 steps on an L4.
* Multiply the hours by the compute units per hour shown in *Resources* to compare GPUs.
* If GPU memory stays far below the total, `training.micro_batch_size: 4` (in 5.1, new id) runs faster.

In [ ]:
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU: Runtime > Change runtime type > GPU, then run section 1 again")
gpu_short = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                           capture_output=True, text=True).stdout.strip().split()[-1]
SMOKE = os.path.join(RUNS, f"_smoke_{EXPERIMENT_ID}_{gpu_short}")
shutil.rmtree(SMOKE, ignore_errors=True)            # scratch folder: always measure from step 0
mem_log = f"/tmp/mem_{gpu_short}.csv"
monitor = subprocess.Popen(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits",
                            "-lms", "500", "-f", mem_log])
t0 = time.time()
!python scripts/train.py --config "{TRAIN_CONFIG}" --data-root "{DATA_ROOT}" --run-dir "{SMOKE}" --prithvi-weights "{WEIGHTS}" --steps 20
wall = time.time() - t0
monitor.terminate(); monitor.wait()

history = pd.read_csv(os.path.join(SMOKE, "training_history.csv"))
print(history[["step", "train_WCE", "grad_norm", "seconds"]].tail(5).to_string(index=False))
sec = history["seconds"].iloc[5:].median()         # skip warm-up steps
steps = load_config(TRAIN_CONFIG)["training"]["maximum_steps"]
print(f"\n{gpu_short}: {sec:.2f} s/step -> about {sec * steps / 3600:.1f} h for {steps} steps (+ validation and checkpoints)")
print(f"setup + loading + one checkpoint save: {wall - history['seconds'].sum():.0f} s")
if os.path.getsize(mem_log):
    mem = pd.read_csv(mem_log, header=None, skipinitialspace=True)
    print(f"peak GPU memory: {mem[0].max():,} / {mem[1].max():,} MiB (memory reserved by PyTorch, an upper bound)")

### 5.5 Full run

Each validation prints `step N: macro-F1 ..., macro-IoU ..., gate ...`. If the session stops, run this cell again
(after section 1): training resumes from `latest_resume.pt` (saved every 78 steps). Keep the browser tab open.

Drive space per run: `best_any.pt` about 1.3 GB and `latest_resume.pt` about 3.9 GB (weights plus optimiser state).
When a run is finished, `latest_resume.pt` can be deleted.

In [ ]:
!python scripts/train.py --config "{TRAIN_CONFIG}" --data-root "{DATA_ROOT}" --run-dir "{RUN_DIR}" --prithvi-weights "{WEIGHTS}"

### 5.6 Results

* `macro-F1` is the mean F1 over all classes, each class counting equally. Rare classes therefore move it a lot:
  a few hundred pixels of a rare class can change it by 0.01-0.02 while the map barely changes.
* `gate` (acceptance) = every class predicted and recall >= 0.05 on the rare classes. Runs that never pass keep
  `best_any.pt` only and their maps are exploratory.
* Reference values (Xuan Thuy, same split): U-Net 6 bands from scratch 0.415; P2 0.443; P3 0.448 and 0.429 (two runs).
* The per-class table below is from the **last** validation. For the per-class scores of the best checkpoint use section 6.

In [ ]:
done = os.path.join(RUN_DIR, "TRAINING_COMPLETE.json")
print(json.load(open(done if os.path.exists(done) else os.path.join(RUN_DIR, "TRAINING_PROGRESS.json"))))
val = pd.read_csv(os.path.join(RUN_DIR, "validation_history.csv"))
hist = pd.read_csv(os.path.join(RUN_DIR, "training_history.csv"))
best = val.loc[val["macro_F1_11"].idxmax()]
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for col in ["macro_F1_11", "macro_IoU_11", "OA"]:
    axes[0].plot(val["step"], val[col], marker="o", label=col)
axes[0].axvline(best["step"], ls="--", c="grey"); axes[0].legend(); axes[0].set_xlabel("step")
axes[0].set_title(f"validation; best macro-F1 {best['macro_F1_11']:.4f} at step {int(best['step'])}")
axes[1].plot(hist["step"], hist["train_WCE"], alpha=0.3); axes[1].plot(hist["step"], hist["train_WCE"].rolling(50).mean())
axes[1].set_title("training loss (WCE), 50-step mean"); axes[1].set_xlabel("step")
plt.tight_layout()
print(pd.read_csv(os.path.join(RUN_DIR, "latest_validation_per_class.csv")).round(3).to_string(index=False))

In [ ]:
# Map of the target date written at the end of training. For other dates: section 4 with
# CHECKPOINT = os.path.join(RUN_DIR, "best_any.pt")
run_map = sorted(glob.glob(os.path.join(RUN_DIR, "map", "*_classes.tif")))
if run_map:
    show_map(run_map[-1], load_config(TRAIN_CONFIG)["class_map"], title=f"{EXPERIMENT_ID}: {os.path.basename(run_map[-1])}")
else:
    print("No map yet: the run is not finished.")

## 6. Score and compare checkpoints on the validation band

Runs the validation of training on any checkpoint, including old-repository ones, and puts the per-class scores side
by side. Use it to see which classes make two runs differ, or to check that the scoring code reproduces a known value
(the old P3 `best_any.pt` scores macro-F1 0.4484, OA 0.814, 10 classes predicted).

All checkpoints must have the architecture of `EVAL_CONFIG` (same bands and number of dates). A scratch run folder is
used, so no existing run is touched.

In [ ]:
# EDIT -----------------------------------------------------------------------------------------
EVAL_CONFIG = "configs/p3_single_date.json"           # defines the data, the split and the architecture
EVAL_DATA_ROOT = DATA_ROOT
CHECKPOINTS = {
    "P3 old repo": os.path.join(BASE, "runs", "P3_single_sep21_UNet_WCE_REPRO", "best_any.pt"),
    "P3 new run": os.path.join(RUNS, "p3_repro", "best_any.pt"),
}
# ----------------------------------------------------------------------------------------------
import torch
from prithvi_unet.training import Trainer
from prithvi_unet.inference.maps import validate
from prithvi_unet.inference.predict import load_any_checkpoint

EVAL_DIR = os.path.join(RUNS, "_eval_scratch")
shutil.rmtree(EVAL_DIR, ignore_errors=True)          # scratch folder: rebuilt each time, never locked to old versions
evaluator = Trainer(EVAL_CONFIG, EVAL_DATA_ROOT, EVAL_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
summaries, per_class = {}, {}
for name, path in CHECKPOINTS.items():
    model = evaluator.load_model(pretrained=False)
    load_any_checkpoint(path, model)
    model.to(device)
    summary, table = validate(model, evaluator.scene, evaluator.val_windows, device)
    summaries[name] = {k: summary[k] for k in ["macro_F1_11", "macro_IoU_11", "OA", "predicted_classes", "acceptance_passed"]}
    per_class[name] = table.set_index("class")
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
print(pd.DataFrame(summaries).T.round(4))
first = next(iter(per_class.values()))
table = pd.concat({"support": first["support"], **{f"F1 {n}": t["F1"] for n, t in per_class.items()},
                   **{f"recall {n}": t["recall"] for n, t in per_class.items()}}, axis=1)
print(table.round(3).to_string())

## 7. Download Sentinel-2 images for another area or period

`scripts/export_sentinel2.py` searches Microsoft Planetary Computer, keeps one scene per date (scene cloud cover below
`--max-cloud` %, footprint covering at least `--min-coverage` % of the area), and writes one GeoTIFF per date with
everything the pipeline needs: band names, `acquisition_date`, `s2_processing_baseline`, `boa_add_offset`.
Always run with `--dry-run` first to see which dates would be exported.

Limits: scene cloud cover is a whole-scene number, not a per-pixel mask: check every image in 3.4.

Put the area as a GeoJSON or shapefile on Drive (for example drawn in QGIS and saved as `aoi/my_area.geojson`).

In [ ]:
# EDIT -----------------------------------------------------------------------------------------
AOI = os.path.join(BASE, "aoi", "my_area.geojson")
START, END = "2026-10-01", "2026-10-31"
EXPORT_OUT = os.path.join(EXPORTS, "my_area")
MAX_CLOUD = 10                                         # %
# ----------------------------------------------------------------------------------------------
!python scripts/export_sentinel2.py --aoi "{AOI}" --start {START} --end {END} --out "{EXPORT_OUT}" --max-cloud {MAX_CLOUD} --dry-run

In [ ]:
# Export for real (after reading the dry-run list), then check the images in section 3 with
# IMAGES = sorted(glob.glob(os.path.join(EXPORT_OUT, "*.tif")))
!python scripts/export_sentinel2.py --aoi "{AOI}" --start {START} --end {END} --out "{EXPORT_OUT}" --max-cloud {MAX_CLOUD}

## 8. Utilities

* **Convert an old-repository checkpoint** to this package's layout. Not required (old checkpoints are converted on
  the fly when loaded) but the converted file loads faster and carries a format tag. The script loads the result
  into the model with `strict=True`, which proves that every weight was mapped.
* **Shapes and parameters** of any config, for example after changing `decoder_channels` or `bands`.

In [ ]:
# EDIT -----------------------------------------------------------------------------------------
OLD_CHECKPOINT = os.path.join(BASE, "runs", "P3_single_sep21_UNet_WCE_REPRO", "best_any.pt")
OLD_CONFIG = "configs/p3_single_date.json"
CONVERTED = os.path.join(RUNS, "converted", "P3_old_best_any.pt")
# ----------------------------------------------------------------------------------------------
os.makedirs(os.path.dirname(CONVERTED), exist_ok=True)
!python scripts/convert_legacy_checkpoint.py "{OLD_CHECKPOINT}" "{CONVERTED}" --config {OLD_CONFIG}

In [ ]:
SHAPES_CONFIG = TRAIN_CONFIG if "TRAIN_CONFIG" in globals() else "configs/p3_single_date.json"   # EDIT if needed
!python scripts/trace_shapes.py --config "{SHAPES_CONFIG}"

## 9. End the session

Writes any file still buffered to Drive, then releases the machine so it stops using compute units.
The notebook disconnects; outputs already shown stay in the notebook.

In [ ]:
if ON_COLAB:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    runtime.unassign()